In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 13.4 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import optuna

# =====================================================
# Load data
# =====================================================

data = torch.load("/content/drive/MyDrive/gsc_mfcc_40x98.pt")

X_train = data["X_train"].unsqueeze(1).float()
y_train = data["y_train"]

X_val = data["X_val"].unsqueeze(1).float()
y_val = data["y_val"]

# =====================================================
# Normalize
# =====================================================

mean = X_train.mean()
std = X_train.std()

X_train = (X_train - mean) / std
X_val   = (X_val - mean) / std

# =====================================================
# Dataset
# =====================================================

train_ds = torch.utils.data.TensorDataset(X_train, y_train)
val_ds   = torch.utils.data.TensorDataset(X_val, y_val)

# =====================================================
# Your Model (Dynamic FC Input)
# =====================================================

class KWS_CNN(nn.Module):
    def __init__(self, input_shape, num_classes=12, dropout_prob=0.3):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels=1,
            out_channels=8,
            kernel_size=(13, 3),
            stride=(1, 1),
            padding=(0, 1)
        )

        self.pool1 = nn.MaxPool2d(kernel_size=(1, 2))

        self.conv2 = nn.Conv2d(
            in_channels=8,
            out_channels=32,
            kernel_size=(1, 3),
            stride=(1, 1),
            padding=(0, 1)
        )

        self.dropout = nn.Dropout(dropout_prob)

        # ==========================================
        # Dynamically calculate FC input size
        # ==========================================
        with torch.no_grad():
            dummy = torch.zeros(1, *input_shape)
            x = F.relu(self.conv1(dummy))
            x = self.pool1(x)
            x = F.relu(self.conv2(x))
            fc_input = x.numel()

        self.fc = nn.Linear(fc_input, num_classes)

    def forward(self, x):

        x = F.relu(self.conv1(x))
        x = self.pool1(x)

        x = F.relu(self.conv2(x))
        x = self.dropout(x)

        x = torch.flatten(x, start_dim=1)

        x = self.fc(x)

        return x

# =====================================================
# Device
# =====================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =====================================================
# Input Shape from MFCC Automatically
# =====================================================

input_shape = X_train.shape[1:]   # e.g. (1, 40, 98)

# =====================================================
# Optuna Objective
# =====================================================

def objective(trial):

    # ---------------- Hyperparameters ----------------
    lr = trial.suggest_float("lr", 1e-4, 1e-3, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    dropout = trial.suggest_float("dropout", 0.2, 0.6)
    label_smoothing = trial.suggest_float("label_smoothing", 0.0, 0.15)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])

    # ---------------- Loaders ----------------
    train_loader = torch.utils.data.DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )

    val_loader = torch.utils.data.DataLoader(
        val_ds,
        batch_size=batch_size,
        num_workers=2,
        pin_memory=True
    )

    # ---------------- Model ----------------
    model = KWS_CNN(
        input_shape=input_shape,
        dropout_prob=dropout
    ).to(device)

    # ---------------- Loss ----------------
    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)

    # ---------------- Optimizer ----------------
    optimizer = optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    # ---------------- Scheduler ----------------
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='max',
        factor=0.5,
        patience=2
    )

    # ---------------- Training ----------------
    epochs = 10
    best_val_acc = 0

    for epoch in range(epochs):

        # ===== Train =====
        model.train()

        for x, y in train_loader:

            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            optimizer.zero_grad()

            outputs = model(x)

            loss = criterion(outputs, y)

            loss.backward()

            optimizer.step()

        # ===== Validation =====
        model.eval()

        correct = 0
        total = 0

        with torch.no_grad():

            for x, y in val_loader:

                x = x.to(device, non_blocking=True)
                y = y.to(device, non_blocking=True)

                outputs = model(x)

                preds = outputs.argmax(1)

                correct += (preds == y).sum().item()
                total += y.size(0)

        val_acc = correct / total

        scheduler.step(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc

        # report to optuna
        trial.report(val_acc, epoch)

        # pruning
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return best_val_acc

# =====================================================
# Run Study
# =====================================================

study = optuna.create_study(direction="maximize")

study.optimize(objective, n_trials=30)

# =====================================================
# Best Result
# =====================================================

print("\n✅ Best Trial")
print("Best Accuracy:", study.best_trial.value)

print("\nBest Hyperparameters:")
for key, value in study.best_trial.params.items():
    print(f"{key}: {value}")

[I 2026-04-01 16:48:42,938] A new study created in memory with name: no-name-f6f19f4c-19e1-4ff8-a637-f9b8af8a00cc
[I 2026-04-01 16:50:43,473] Trial 0 finished with value: 0.7817142857142857 and parameters: {'lr': 0.00045598382441241407, 'weight_decay': 0.00036710095482063595, 'dropout': 0.3436962307462668, 'label_smoothing': 0.023593176276988075, 'batch_size': 16}. Best is trial 0 with value: 0.7817142857142857.
[I 2026-04-01 16:51:45,337] Trial 1 finished with value: 0.7933333333333333 and parameters: {'lr': 0.0009549783353431774, 'weight_decay': 1.2818763748439466e-06, 'dropout': 0.5041284851744658, 'label_smoothing': 0.00910811077703048, 'batch_size': 32}. Best is trial 1 with value: 0.7933333333333333.
[I 2026-04-01 16:53:37,360] Trial 2 finished with value: 0.7895238095238095 and parameters: {'lr': 0.000642950509399946, 'weight_decay': 2.8468095934927505e-05, 'dropout': 0.5183999047435497, 'label_smoothing': 0.03610600288025626, 'batch_size': 16}. Best is trial 1 with value: 0.793


✅ Best Trial
Best Accuracy: 0.804952380952381

Best Hyperparameters:
lr: 0.0009987113423775241
weight_decay: 5.453212400544388e-06
dropout: 0.4377582849751788
label_smoothing: 0.0015596540153027501
batch_size: 32


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import optuna

# =====================================================
# Load data
# =====================================================

data = torch.load("/content/drive/MyDrive/gsc_mfcc_40x98.pt")

X_train = data["X_train"].unsqueeze(1).float()
y_train = data["y_train"]

X_val = data["X_val"].unsqueeze(1).float()
y_val = data["y_val"]

# =====================================================
# Normalize
# =====================================================

mean = X_train.mean()
std = X_train.std()

X_train = (X_train - mean) / std
X_val   = (X_val - mean) / std

# =====================================================
# Dataset
# =====================================================

train_ds = torch.utils.data.TensorDataset(X_train, y_train)
val_ds   = torch.utils.data.TensorDataset(X_val, y_val)

# =====================================================
# Dynamic Model
# =====================================================

class KWS_CNN(nn.Module):
    def __init__(self, input_shape, num_classes,
                 conv1_out, conv2_out, dropout_prob):
        super().__init__()

        freq_bins = input_shape[1]

        self.conv1 = nn.Conv2d(
            1,
            conv1_out,
            kernel_size=(freq_bins,3),
            padding=(0,1)
        )

        self.pool1 = nn.MaxPool2d((1,2))

        self.conv2 = nn.Conv2d(
            conv1_out,
            conv2_out,
            kernel_size=(1,3),
            padding=(0,1)
        )

        self.dropout = nn.Dropout(dropout_prob)

        with torch.no_grad():
            dummy = torch.zeros(1, *input_shape)

            x = F.relu(self.conv1(dummy))
            x = self.pool1(x)

            x = F.relu(self.conv2(x))

            fc_input = x.numel()

        self.fc = nn.Linear(fc_input, num_classes)

    def forward(self, x):

        x = F.relu(self.conv1(x))
        x = self.pool1(x)

        x = F.relu(self.conv2(x))

        x = self.dropout(x)

        x = torch.flatten(x,1)

        return self.fc(x)

# =====================================================
# Device
# =====================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

input_shape = X_train.shape[1:]
num_classes = len(torch.unique(y_train))

# =====================================================
# Optuna Objective
# =====================================================

def objective(trial):

    # ---------------- Hyperparameters ----------------
    conv1_out = trial.suggest_categorical("conv1_out", [16, 24, 32, 48])

    conv2_out = trial.suggest_categorical("conv2_out", [8, 16, 24, 32])

    dropout = trial.suggest_float("dropout", 0.2, 0.6)

    lr = trial.suggest_float("lr", 1e-4, 5e-3, log=True)

    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)

    label_smoothing = trial.suggest_float("label_smoothing", 0.0, 0.15)

    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])

    # ---------------- Loaders ----------------
    train_loader = torch.utils.data.DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )

    val_loader = torch.utils.data.DataLoader(
        val_ds,
        batch_size=batch_size,
        num_workers=2,
        pin_memory=True
    )

    # ---------------- Model ----------------
    model = KWS_CNN(
        input_shape=input_shape,
        num_classes=num_classes,
        conv1_out=conv1_out,
        conv2_out=conv2_out,
        dropout_prob=dropout
    ).to(device)

    # ---------------- Loss ----------------
    criterion = nn.CrossEntropyLoss(
        label_smoothing=label_smoothing
    )

    # ---------------- Optimizer ----------------
    optimizer = optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    # ---------------- Scheduler ----------------
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='max',
        factor=0.5,
        patience=2
    )

    # ---------------- Training ----------------
    best_val_acc = 0

    for epoch in range(10):

        model.train()

        for x, y in train_loader:

            x = x.to(device)
            y = y.to(device)

            optimizer.zero_grad()

            outputs = model(x)

            loss = criterion(outputs, y)

            loss.backward()

            optimizer.step()

        # Validation
        model.eval()

        correct = 0
        total = 0

        with torch.no_grad():

            for x, y in val_loader:

                x = x.to(device)
                y = y.to(device)

                outputs = model(x)

                preds = outputs.argmax(1)

                correct += (preds == y).sum().item()
                total += y.size(0)

        val_acc = correct / total

        scheduler.step(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc

        trial.report(val_acc, epoch)

        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return best_val_acc

# =====================================================
# Run Study
# =====================================================

study = optuna.create_study(direction="maximize")

study.optimize(objective, n_trials=30)

# =====================================================
# Results
# =====================================================

print("\n✅ Best Accuracy:", study.best_trial.value)

print("\nBest Params:")
for k,v in study.best_trial.params.items():
    print(k, v)

[I 2026-04-01 17:27:36,096] A new study created in memory with name: no-name-7fc5d1bf-12ee-4bb5-a9d1-0cc5fbaa3ad2
[I 2026-04-01 17:28:39,096] Trial 0 finished with value: 0.8034285714285714 and parameters: {'conv1_out': 16, 'conv2_out': 16, 'dropout': 0.218362593898489, 'lr': 0.0004547711313434782, 'weight_decay': 0.000675031534276569, 'label_smoothing': 0.05178460058536824, 'batch_size': 32}. Best is trial 0 with value: 0.8034285714285714.
[I 2026-04-01 17:29:40,981] Trial 1 finished with value: 0.8392380952380952 and parameters: {'conv1_out': 16, 'conv2_out': 32, 'dropout': 0.44081523700409075, 'lr': 0.0004148378163532324, 'weight_decay': 1.0722579894385448e-06, 'label_smoothing': 0.07129201109711128, 'batch_size': 32}. Best is trial 1 with value: 0.8392380952380952.
[I 2026-04-01 17:31:33,807] Trial 2 finished with value: 0.8733333333333333 and parameters: {'conv1_out': 32, 'conv2_out': 24, 'dropout': 0.34116113722464225, 'lr': 0.002434651645507094, 'weight_decay': 1.187713852679567


✅ Best Accuracy: 0.8950476190476191

Best Params:
conv1_out 48
conv2_out 32
dropout 0.27651563138245827
lr 0.0019760425054724754
weight_decay 0.00018723454455808885
label_smoothing 0.1408435657900889
batch_size 32
